In [1]:
# Weapon class

import threading
from dataclasses import dataclass, field

# Custom exception class
class WeaponError(Exception):
    """Base class for weapon-related exceptions."""
    def __init__(self, message: str):
        super().__init__(message)

@dataclass(frozen=True)
class Weapon:
    _ammo_max: int = field(default=5)
    _ammo: int = field(default=_ammo_max)
    _damage: float = field(default=1.0)
    _loading: bool = field(default=False)
    _load_delay: float = field(default=5.0)
    _lock: threading.Lock = field(default_factory=threading.Lock, init=False)

    def __post_init__(self):
        if self._ammo > self._ammo_max:
            raise ValueError("Ammo cannot exceed ammo_max")

    @property
    def ammo(self) -> int:
        return self._ammo

    # @ammo.setter
    # def ammo(self, value):
    #     # return Weapon(_ammo=value, _ammo_max=self._ammo_max, _damage=self._damage, _loading=self._loading, _load_delay=self._load_delay)
    #     self._ammo = value

    @property
    def ammo_max(self) -> int:
        return self._ammo_max

    @property
    def damage(self) -> float:
        return self._damage

    @property
    def loading(self) -> bool:
        return self._loading

    @property
    def load_delay(self) -> float:
        return self._load_delay

    @property
    def is_empty(self) -> bool:
        return self.ammo == 0

    @property
    def can_reload(self) -> bool:
        return not self._loading and self._ammo < self._ammo_max

    def shot(self) -> 'Weapon':
        if self.is_empty:
            raise WeaponError("Cannot shoot: Out of ammo, current ammo: {}".format(self.ammo))
        
        with self._lock:
            return Weapon(_ammo=self._ammo - 1, _ammo_max=self._ammo_max, _damage=self._damage,
                          _loading=self._loading, _load_delay=self._load_delay)

    def reload(self) -> 'Weapon':
        if self.loading:
            raise WeaponError("Already loading. Wait for the current reload to finish.")
        
        with self._lock:
            return Weapon(_ammo=self._ammo_max, _ammo_max=self._ammo_max, _damage=self._damage,
                          _loading=True, _load_delay=self._load_delay)

    def __hash__(self):
        return hash((self._ammo, self._ammo_max, self._damage, self._loading, self._load_delay))


In [2]:
# Shield class

from dataclasses import dataclass, field
import threading
from typing import Optional

@dataclass(frozen=True)
class Shield:
    level: Optional[float] = field(default=None)  # Input shield level
    shield_max: int = field(default=5)  # Default max shield level
    repair_delay: int = field(default=5)   # Default repair delay
    _is_repairing: threading.Event = field(default_factory=threading.Event, init=False)
    lock: threading.Lock = field(default_factory=threading.Lock, init=False)

    def __post_init__(self):
        # Normalize level to be >= 0 and <= shield_max
        if self.level is None:
            object.__setattr__(self, 'level', self.shield_max)
        else:
            normalized_level = min(max(self.level, 0), self.shield_max)
            object.__setattr__(self, 'level', normalized_level)

        object.__setattr__(self, '_is_repairing', False)  # Initialize repairing status

    @property
    def is_repairing(self) -> bool:
        """Check if the shield is currently being repaired."""
        with self.lock:
            return self._is_repairing

    def repair_shield(self) -> None:
        """Initiate a repair of the shield after a delay."""
        with self.lock:
            current_level = self.level if self.level is not None else 0
            if current_level < self.shield_max and not self._is_repairing:
                object.__setattr__(self, '_is_repairing', True)
                threading.Timer(self.repair_delay, self._finish_repair).start()

    def _finish_repair(self) -> None:
        """Finish repairing the shield."""
        with self.lock:
            object.__setattr__(self, 'level', self.shield_max)
            object.__setattr__(self, '_is_repairing', False)

    def damage_shield(self, damage: float) -> 'Shield':
        """Apply damage to the shield, returning a new Shield instance with updated level."""
        with self.lock:
            current_level = self.level if self.level is not None else 0
            new_level = max(current_level - damage, 0)
            return Shield(level=new_level, shield_max=self.shield_max, repair_delay=self.repair_delay)

    def __eq__(self, other: object) -> bool:
        """Check if two shields are equal based on their levels and max values."""
        if not isinstance(other, Shield):
            return NotImplemented
        return (self.level == other.level and 
                self.shield_max == other.shield_max)

    def __hash__(self) -> int:
        """Return the hash based on shield level and max."""
        return hash((self.level, self.shield_max))

    def __str__(self) -> str:
        return (f"Shield Level: {self.level}/{self.shield_max}, "
                f"Repairing: {self.is_repairing}")

    def __lt__(self, other: 'object') -> bool:
        """Compare shields based on their shield levels."""
        if not isinstance(other, Shield):
            return NotImplemented
        s = self.level if self.level is not None else 0
        o = other.level if other.level is not None else 0
        return s < o
        

In [3]:
# FuelTank class

from dataclasses import dataclass, field
from math import inf


@dataclass(frozen=True)
class FuelTank:
    level: float = field(default=-inf)
    cost: float = field(default=0.25)
    volume: float = field(default=100)

    def __post_init__(self):
        if self.level > self.volume:
            raise ValueError("Fuel tank level cannot be greater than volume")
        if self.level < 0:
            object.__setattr__(self, "level", self.volume)
        if self.cost <= 0 or self.volume <= 0:
            raise ValueError("Cost and volume must be positive.")

    def drop_fuel(self, distance: float):
        new_level = self.level - (distance * self.cost)
        if new_level < 0:
            raise ValueError("Fuel tank level cannot be negative")
        return FuelTank(level=new_level, cost=self.cost, volume=self.volume)

    def refuel(self):
        # takes time can stop mid-way
        ...
        if self.is_full():
            raise ValueError("Fuel tank is full")

        return FuelTank(level=self.volume, cost=self.cost, volume=self.volume)

    def is_empty(self):
        return self.level == 0

    def is_full(self):
        return self.level == self.volume